# LG-01: LangGraph 基础与图构建> **阶段**: LG-01 | **难度**: 入门 | **预计时长**: 3-4 小时## 学习目标- 理解 LangGraph 的核心理念- 掌握 State、Node、Edge 三大基石概念- 能够独立构建并运行一个简单的 StateGraph- 理解 Reducer 机制与状态更新原理- 了解 Channel 底层机制与 Runtime Context

In [ ]:
# 安装依赖（如未安装请取消注释）# !pip install -U langgraph langchain langchain-openai

## 1. 核心概念：为什么用"图"替代"链"？传统的 LangChain Chain 是**线性**的：A → B → CLangGraph 是**图结构**的：节点可以循环、分支、并行```Chain:  START → A → B → C → ENDGraph:  START → A → [条件判断] → B → D → END                     └────────→ C ────┘```**五个核心概念**：State、Node、Edge、Conditional Edge、compile/invoke/stream用一句话概括：> **State 是数据，Node 是处理，Edge 是路由，compile 是生成，invoke/stream 是运行。**

## 2. State —— 图里的共享数据State 是一个 `TypedDict`，定义了图中所有节点共享的数据结构。**关键点**：- 每个节点接收**完整 State**- 节点返回**增量更新**- `Annotated[field, reducer]` 决定字段怎么合并**常见坑**：节点内直接修改 state 不会生效！必须通过返回值让框架合并。

In [ ]:
from typing import TypedDict, Annotatedfrom operator import addfrom langchain_core.messages import HumanMessage, AIMessageclass WeatherState(TypedDict):    city: str    intent: str    weather_info: str    response: str    messages: Annotated[list, add]print("State 定义完成!")

### 2.1 深入：Annotated 与 Reducer`Annotated[type, reducer]` 给每个字段附加一条**合并规则**。**默认行为（不写 Annotated）= last-write-wins**。**列表/消息字段必须加 Reducer**，否则每次都被覆盖。

In [ ]:
from langgraph.graph.message import add_messagesfrom langchain_core.messages import AnyMessageclass ChatState(TypedDict):    tags: Annotated[list[str], add]    messages: Annotated[list[AnyMessage], add_messages]print("operator.add: 直接拼接两个列表")print("add_messages: 拼接消息 + 按 ID 去重")

### 2.2 自定义 ReducerReducer 本质上是一个函数：`(old_value, new_value) -> merged_value`。必须满足三条规则：1. **纯函数** —— 不能有副作用2. **处理初始空值**3. **返回类型匹配**

In [ ]:
def keep_last_n(n: int):    def _reducer(old: list, new: list) -> list:        return (old + new)[-n:]    return _reducerdef merge_dicts(old: dict, new: dict) -> dict:    return {**old, **new}class CustomState(TypedDict):    history: Annotated[list, keep_last_n(10)]    metadata: Annotated[dict, merge_dicts]print("自定义 Reducer 定义完成")

## 3. Node —— 图里的处理单元Node 就是一个函数：接收 `state`，返回一个 `dict`。- Node 可以是任何 Python 函数- 返回值必须是 dict

In [ ]:
from datetime import datetimedef parse_intent(state: WeatherState) -> dict:    msg = state["messages"][-1] if state["messages"] else None    text = msg.content if hasattr(msg, "content") else str(msg) if msg else ""    if "天气" in text or "温度" in text:        intent = "weather"    elif "时间" in text or "几点" in text:        intent = "time"    else:        intent = "other"    return {"intent": intent, "messages": [f"意图识别: {intent}"]}def extract_city(state: WeatherState) -> dict:    msg = state["messages"][-1] if state["messages"] else None    text = msg.content if hasattr(msg, "content") else str(msg) if msg else ""    city = "北京" if "北京" in text else "上海" if "上海" in text else ""    return {"city": city, "messages": [f"城市提取: {city}"]}def query_weather(state: WeatherState) -> dict:    city = state.get("city", "未知")    weather_data = {"北京": {"temp": 25, "condition": "晴"}, "上海": {"temp": 28, "condition": "多云"}}    data = weather_data.get(city, {"temp": 0, "condition": "未知"})    return {"weather_info": f"{data["condition"]}，{data["temp"]}°C", "messages": ["天气查询完成"]}def format_weather_reply(state: WeatherState) -> dict:    city = state.get("city", "未知")    weather = state.get("weather_info", "")    return {"response": f"{city}今天{weather}，祝您出行愉快！", "messages": ["格式化回复完成"]}def get_time(state: WeatherState) -> dict:    now = datetime.now().strftime("%H:%M")    return {"response": f"现在是 {now}", "messages": [f"时间查询: {now}"]}def fallback_reply(state: WeatherState) -> dict:    return {"response": "抱歉，我暂时只能回答天气和时间相关的问题。", "messages": ["兜底回复"]}print("所有节点函数已定义")

### 3.1 必答问题：一个个节点走，跟我直接写函数有什么区别？| 需求 | 纯函数 | LangGraph ||------|--------|-----------|| 基础 WeatherBot | 10 行 | 30 行 || + 重试 | 35 行 (↑250%) | 40 行 (↑33%) || + 并行查询 | 60+ 行 (↑500%) | 55 行 (↑83%) || + 人机协作 | 架构重写 | 70 行 (↑133%) |> **一句话**：纯函数做"一次性的事"，LangGraph 做"有状态、可编排、可恢复的事"

## 4. Edge —— 普通边与条件边### 4.1 普通边：确定性的连接普通边：A 做完必定到 B。

In [ ]:
from langgraph.graph import StateGraph, START, ENDbuilder = StateGraph(WeatherState)builder.add_node("parse_intent", parse_intent)builder.add_node("extract_city", extract_city)builder.add_node("query_weather", query_weather)builder.add_node("format_weather_reply", format_weather_reply)builder.add_node("get_time", get_time)builder.add_node("fallback_reply", fallback_reply)builder.add_edge(START, "parse_intent")builder.add_edge("extract_city", "query_weather")builder.add_edge("query_weather", "format_weather_reply")builder.add_edge("format_weather_reply", END)builder.add_edge("get_time", END)builder.add_edge("fallback_reply", END)print("普通边已添加")

### 4.2 条件边：图的灵魂条件边让图能根据 State 的内容，动态决定下一步走哪个节点。**Router 函数**：接收 `state`，返回一个**字符串**，必须是目标节点的名字。

In [ ]:
def route_by_intent(state: WeatherState) -> str:    intent = state["intent"]    print(f"[路由决策] 意图={intent}")    if intent == "weather":        return "extract_city"    elif intent == "time":        return "get_time"    else:        return "fallback_reply"def route_after_city(state: WeatherState) -> str:    if state.get("city"):        return "query_weather"    return "fallback_reply"builder.add_conditional_edges("parse_intent", route_by_intent, {"extract_city": "extract_city", "get_time": "get_time", "fallback_reply": "fallback_reply"})builder.add_conditional_edges("extract_city", route_after_city, {"query_weather": "query_weather", "fallback_reply": "fallback_reply"})print("条件边已添加")

## 5. compile —— 生成可执行的图`compile()` 做了什么？- 类型检查- 边验证- 生成可执行对象**常见坑**：不 compile 直接 invoke 会报错。compile 是必需的。

In [ ]:
graph = builder.compile()print("图编译成功！")print("weather 分支: START -> parse_intent -> extract_city -> query_weather -> format_weather_reply -> END")print("time 分支:    START -> parse_intent -> get_time -> END")print("other 分支:   START -> parse_intent -> fallback_reply -> END")

## 6. 图的可视化三种查看方式：1. Jupyter Notebook 里直接显示2. 保存成文件3. LangGraph Studio

In [ ]:
from IPython.display import Image, displaypng_bytes = graph.get_graph().draw_mermaid_png()display(Image(data=png_bytes))print("上图展示了 WeatherBot 的图结构")

## 7. invoke vs stream —— 两种运行方式**invoke**：直接给最终结果。**stream**：逐个节点输出中间状态，适合调试和观察流程。

In [ ]:
print("=" * 60)print("测试 1: 查询天气")print("=" * 60)result = graph.invoke({"messages": [HumanMessage("北京今天天气怎么样？")], "intent": "", "city": "", "weather_info": "", "response": ""})print(f"最终回复: {result["response"]}")print(f"执行日志: {result["messages"]}")

In [ ]:
print("=" * 60)print("测试 2: 使用 stream 观察中间状态")print("=" * 60)for chunk in graph.stream({"messages": [HumanMessage("现在几点了？")], "intent": "", "city": "", "weather_info": "", "response": ""}):    for node_name, node_state in chunk.items():        print(f"节点: {node_name}")

In [ ]:
test_queries = ["北京今天天气怎么样？", "现在几点了？", "讲个笑话"]for query in test_queries:    print(f"用户: {query}")    result = graph.invoke({"messages": [HumanMessage(query)], "intent": "", "city": "", "weather_info": "", "response": ""})    print(f"Bot: {result["response"]}")

## 8. MessagesState —— 聊天场景的预制模板如果只是为了做聊天机器人，LangGraph 已经预制了一个最常用 State 模板：

In [ ]:
from langgraph.graph import MessagesStateclass ChatBotState(MessagesState):    city: str    intent: strprint("MessagesState 预置字段: messages: list[Message]")

## 9. Channel 深入（延伸阅读）LangGraph Channels（通道）是底层架构的核心组件，负责在 Graph 节点之间**传递和存储数据**。### 9.1 Channel 类型详解| Channel 类型 | 更新策略 | 步骤内多值 | 跨步骤保持 | 典型用途 ||-------------|---------|-----------|-----------|---------|| **LastValue** | 覆盖 | 仅一个 | 保持 | 单一结果、状态标记 || **Topic** | 列表累积 | 允许 | 可配置 | 事件、日志、消息队列 || **BinaryOperatorAggregate** | 聚合函数 | 允许 | 保持 | 计数器、累加、拼接 || **EphemeralValue** | 覆盖 | 仅一个 | 清空 | 临时通知、一次性信号 || **AnyValue** | 覆盖（需相同） | 允许 | 保持 | 冗余更新、一致性检查 |

In [ ]:
class LogState(TypedDict):    logs: Annotated[list, add]    current_step: str    error_count: Annotated[int, add]def step_1(state: LogState):    return {"logs": ["Step 1"], "current_step": "step_1", "error_count": 0}def step_2(state: LogState):    return {"logs": ["Step 2"], "current_step": "step_2", "error_count": 1}log_builder = StateGraph(LogState)log_builder.add_node("step_1", step_1)log_builder.add_node("step_2", step_2)log_builder.add_edge(START, "step_1")log_builder.add_edge("step_1", "step_2")log_builder.add_edge("step_2", END)log_graph = log_builder.compile()log_result = log_graph.invoke({"logs": [], "current_step": "", "error_count": 0})print(f"logs: {log_result["logs"]}")print(f"error_count: {log_result["error_count"]}")

In [ ]:
class AggregationState(TypedDict):    total_score: Annotated[int, add]    sources: Annotated[list, add]def source_a(state): return {"total_score": 85, "sources": ["A"]}def source_b(state): return {"total_score": 92, "sources": ["B"]}agg_builder = StateGraph(AggregationState)agg_builder.add_node("a", source_a)agg_builder.add_node("b", source_b)agg_builder.add_edge(START, "a")agg_builder.add_edge(START, "b")agg_builder.add_edge("a", END)agg_builder.add_edge("b", END)agg_graph = agg_builder.compile()agg_result = agg_graph.invoke({"total_score": 0, "sources": []})print(f"total_score: {agg_result["total_score"]}")

## 10. Runtime Context（延伸阅读）LangGraph Runtime 是运行时的上下文机制，为 Graph 执行提供统一的环境信息访问接口。### 10.1 Runtime vs State vs Config| 特性 | State | Runtime Context | RunnableConfig ||------|-------|----------------|----------------|| **用途** | 会话业务数据 | 运行环境配置 | 执行配置参数 || **内容** | 消息、结果、进度 | 用户ID、连接、密钥 | thread_id、metadata || **生命周期** | 动态变化 | 静态不变 | 每次调用传入 || **持久化** | 保存到 checkpoint | 不持久化 | 部分持久化 || **可序列化** | 必须可序列化 | 可以不可序列化 | 可序列化 |

In [ ]:
from dataclasses import dataclassfrom typing import Optional, Any@dataclassclass GraphContext:    user_id: str    tenant_id: Optional[str] = None    user_permissions: list = None    api_keys: dict = None    def __post_init__(self):        if self.user_permissions is None:            self.user_permissions = []        if self.api_keys is None:            self.api_keys = {}from langchain_core.runnables import RunnableConfigdef node_with_context(state: WeatherState, config: RunnableConfig):    user_id = config.get("configurable", {}).get("user_id", "anonymous")    return {"messages": [f"用户 {user_id} 的请求已处理"]}print("Runtime Context 示例完成")

## 11. 常见误区| 坑 | 现象 | 怎么避免 ||----|------|---------|| State 写成普通 dict | 编译通过但运行时报错 | 必须用 TypedDict || Reducer 遗漏 | 列表字段每次被覆盖 | 用 Annotated + Reducer || 节点返回字符串 | 报 TypeError | 节点必须返回 dict || 忘记 import START/END | 找不到符号 | 从 langgraph.graph 导入 || 节点内直接改 state | 修改不生效 | 必须通过返回值合并 || 忘记 compile | 报错没有 invoke | compile 是必需的 || Router 返回未注册节点名 | 运行时找不到 | 确保名字一致 || 一个节点同时有普通边和条件边 | 行为不可预期 | 二选一 |

## 12. 速查表 / Cheat Sheet```pythonfrom typing import TypedDict, Annotatedfrom operator import addfrom langgraph.graph import StateGraph, START, ENDfrom langgraph.graph.message import add_messagesclass State(TypedDict):    messages: Annotated[list, add_messages]    result: str    logs: Annotated[list, add]def my_node(state: State):    return {"result": "hello"}builder = StateGraph(State)builder.add_node("node", my_node)builder.add_edge(START, "node")builder.add_edge("node", END)graph = builder.compile()result = graph.invoke({"messages": [], "result": "", "logs": []})```**口诀**：> 状态是数据，节点是工厂，边是传送带，条件边是分岔口，编译出图纸，stream 看过程。

## 13. 课后练习1. **扩展 WeatherBot**：增加一个节点在查询前验证城市名是否为空2. **修改图结构**：让查询天气失败后重试一次（循环）3. **尝试使用 MessagesState**：重写本案例4. **自定义 Reducer**：实现一个滑动窗口 reducer，只保留最近 5 条消息5. **多分支路由**：增加一个"空气质量"分支---**下一节**: LG-02 Tools 深度掌握